In [24]:
import numpy as np



In [25]:
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))


In [26]:
def knn_predict_batch(X_train, y_train, X_test, k, batch_size=256):
    preds = []

    for start in range(0, len(X_test), batch_size):
        end = start + batch_size
        X_batch = X_test[start:end]

        # squared euclidean distance (no sqrt -> faster, same ranking)
        dists = np.sum((X_batch[:, None] - X_train) ** 2, axis=2)

        # indices of nearest neighbors
        sorted_idx = np.argsort(dists, axis=1)

        # labels of k nearest neighbors
        k_neighbors = y_train[sorted_idx[:, :k]]

        # majority vote per row
        for row in k_neighbors:
            values, counts = np.unique(row, return_counts=True)
            preds.append(values[np.argmax(counts)])

    return np.array(preds)


In [27]:
def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)


In [ ]:
import pandas as pd

train = pd.read_csv("train_multi_class.csv")
test = pd.read_csv("test_multi_class.csv")

print(train.head())
print(test.head())
X = train.drop(columns=['target']).values
y = train['target'].values
X_test=test.values


   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
0  27.429698  18.569216  22.457234  33.104879  33.350398  29.937626   
1  37.826448  16.137015  42.383106  45.230917  34.458868  22.003227   
2  23.291904  46.088875  24.247106  19.485180  17.768651  44.653252   
3  37.508083  41.227080  27.673211  23.439315  24.709310  32.663000   
4  18.465239  28.059842  12.871446  24.083452  35.643811  34.329706   

   feature_6  feature_7    feature_8    feature_9  ...  feature_31  \
0  40.707646  37.405466  1060.997693  1093.648897  ...   58.856930   
1  46.884453  36.645896  1010.465824  1107.568316  ...   64.205160   
2  15.362741  30.410329   925.832071   933.016158  ...   39.307505   
3  18.719892  31.260511   969.596824   974.081211  ...   60.657330   
4  44.141067  38.142903   955.414815  1047.003122  ...   40.464344   

   feature_32  feature_33  feature_34   feature_35  feature_36  feature_37  \
0   84.526166   24.684940  -10.883753  5741.962324    0.000285  198.922161

In [ ]:
mask = ~np.isnan(y)
X = X[mask]
y = y[mask]


In [ ]:
np.random.seed(42)
n = X.shape[0]
indices = np.random.permutation(n)

split = int(0.8 * n)
train_idx = indices[:split]
val_idx   = indices[split:]

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]


In [ ]:
import numpy as np

unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))


In [31]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train=(X_train-mean)/std
X_val=(X_val-mean)/std
X_test=(X_test-mean)/std


In [ ]:
#best_k = None
##best_acc = -1

for k in range(4,8):
    y_pred = knn_predict_batch(X_train, y_train, X_val, k, batch_size=256)
    acc = accuracy(y_val, y_pred)
    print(f"K={k}, Accuracy={acc:.3f}")

    if acc > best_acc:
        best_acc = acc
        best_k = k

print("\nBest k:", best_k, " | Best accuracy:", round(best_acc, 3))


K=4, Accuracy=0.700
K=5, Accuracy=0.716
K=6, Accuracy=0.724
K=7, Accuracy=0.733

Best k: 9  | Best accuracy: 0.738


In [ ]:
test_preds=knn_predict_batch(X_train, y_train, X_test, best_k, batch_size=256)



test_preds[:10]


In [38]:
knn_predictions=pd.DataFrame({
 "target":test_preds
})
knn_predictions.to_csv("knn_predictions.csv", index=False)
print("knn_predictions.csv created!")

knn_predictions.csv created!
